# Day2 発展：用語集を使った LoRA 学習と評価

このノートブックは **実行前のキット** です。保存された学習出力・実測スコアは含みません。Colab の **T4 GPU** を選び、上から順に実行すると、データの準備、学習前評価、LoRA 学習、学習後評価、提出用下書き、結果 ZIP まで作成します。

配布ノートブック `notebooks/lora_format_tuning.ipynb` の固定モデル・revision・ライブラリ版・LoRA 対象層を引き継ぎ、学習データを講座用語集に変更しています。基盤モデルは float32・量子化なし、学習は3エポックです。学習の改善は保証せず、**形式遵守、内容、確認要求の適切さ、退行**を実際の出力で確認します。

- 生成モデルに渡すのは同じ system 文と質問だけです。用語集本文・検索結果・正解は渡しません。
- 学習時は配布ノートブックと同じ全トークンの loss を使います。学習と評価の chat template は `enable_thinking=False` で統一します。
- 学習・評価の全入力長を事前確認し、512トークンを超える例があれば切り捨てず停止します。生成上限は256トークンです。
- 学習前後とも `eval()`、貪欲生成、同一の評価設問で比較します。検索は別の比較対象として測定します。
- **GPUを確保できない場合や実行に失敗した場合、学習完了として提出しません。** ランタイムの切断でファイルが消えることがあるため、終わったら最後の ZIP をダウンロードしてください。

実験コードと説明の作成に ChatGPT を使用しています。実行後の出力を自分で確認し、生成された比較・考察を読み、自分で確認した回答例や気づきを追記してください。


## 1. T4 GPU を確認する

Colab の「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選びます。以下が GPU を確認できなければ、後続のセルは実行せず設定を確認してください。


In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
except FileNotFoundError as exc:
    raise RuntimeError("GPUがありません。ランタイムのタイプをT4 GPUへ変更してください。") from exc
if probe.returncode != 0 or not probe.stdout.strip():
    raise RuntimeError("GPUを確認できません。T4 GPUを割り当ててから再実行してください。")
print(probe.stdout.strip())
if "T4" not in probe.stdout:
    print("T4以外のGPUです。実際のGPU名を結果へ記録します。速度の比較では環境差に注意してください。")


## 2. 配布ノートブックと同じ版を導入する

`torch` は Colab 同梱の版を維持し、他の5ライブラリを固定します。配布ノートブックに従い、使用しない `torchao` を取り除きます。ここでは学習もAPI呼び出しも行いません。


In [ ]:
import importlib.metadata
import tempfile

PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
TORCH_VERSION = importlib.metadata.version("torch")
# 依存解決でColab同梱torchが更新されないよう、現在の版を制約する。
constraint = Path(tempfile.gettempdir()) / "day2_colab_torch_constraint.txt"
constraint.write_text(f"torch=={TORCH_VERSION}\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--constraint", str(constraint),
     *[f"{name}=={version}" for name, version in PINS.items()]],
    check=True,
)
assert importlib.metadata.version("torch") == TORCH_VERSION, "Colab同梱torchの版が変化しました。"
VERSIONS = {name: importlib.metadata.version(name) for name in [*PINS, "torch"]}
assert all(VERSIONS[name] == required for name, required in PINS.items())
print(VERSIONS)


## 3. 公開 fork の `lec02` ブランチを取得する

実験コードとデータの出所を揃えるため、`takerunakao/lecture-ai-engineering-2026` の `lec02` を使います。コミット SHA は画面と実験結果の両方に記録します。既存フォルダの変更を強制削除する操作は行いません。


In [ ]:
import os
import json

REPO_URL = "https://github.com/takerunakao/lecture-ai-engineering-2026.git"
BRANCH = "lec02"
REPO = Path("/content/aie2026-day2")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / ".git").is_dir():
        raise RuntimeError("同名の別フォルダがあります。/content/aie2026-day2 を確認してください。")
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("既存フォルダのoriginが指定forkと異なります。内容を確認してください。")
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    current_branch = subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip()
    if dirty or current_branch != BRANCH:
        raise RuntimeError("既存checkoutに未保存の変更があるか、lec02以外のブランチです。結果をダウンロードし、新しいColabセッションで実行してください。")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "FETCH_HEAD"], check=True)
    local_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    public_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
    if local_sha != public_sha:
        raise RuntimeError("ローカルHEADが公開lec02と異なります。既存コミットを消さずに停止します。")
os.chdir(REPO)
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository:", REPO_URL)
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit:", REPO_COMMIT)


## 4. 学習データ・評価設問を準備する

用語集から raw JSONL と、確認要求の負例を含めた学習用 JSONL を作成します。評価用の正解は別ファイルで保持します。データ件数・分割方針は `manifest.json` に記録され、実行時にはデータの SHA-256 も保存されます。

`--dry-run` は設定とデータの確認です。モデルの学習や生成、実測結果の作成は行いません。


In [ ]:
subprocess.run([sys.executable, "scripts/prepare_day2_data.py"], cwd=REPO, check=True)
manifest = json.loads((REPO / "lectures/02/data/manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run([sys.executable, "scripts/run_day2_training.py", "--dry-run"], cwd=REPO, check=True)


## 5. 学習前評価 → LoRA 学習 → 学習後評価

**このセルを実行したときに初めてモデルを取得し、実際に学習します。** 公開モデルの取得に Gemini API キーは不要です。モデル取得・評価・学習の所要時間は、GPUや接続状況によって変わります。

設定は、3エポック、学習率 `2e-4`、LoRA `r=16`、バッチ4・勾配累積2です。124件なら予定の optimizer step は48です。`fp16=False` と `bf16=False` を両方指定し、float32で比較します。

結果は実行ごとに新しいフォルダへ保存します。学習中は10ステップごとにチェックポイントを保存します。失敗時には実行状態を未完了として残し、エラーを表示します。**失敗した場合も、最後のダウンロードセルだけを単独で実行すると、残った部分結果を保存できます。**

コンソールの出力はリアルタイムに表示し、`console.log`にも保存します。失敗時は停止した段階・例外型・詳細を表示します。`traceback.txt`が作られた場合は、最後のZIPにも含めます。


In [ ]:
from datetime import datetime, timezone
import os
import signal
import uuid

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = REPO / "lectures/02/results" / RUN_ID
TRAINING_COMPLETED = False
REPORT_GENERATED = False
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
# runnerは既存のrun_dirを拒否するため、開始前は親フォルダへログを保存する。
CONSOLE_PATH = RUN_DIR.parent / f"{RUN_ID}.console.log"
training_env = dict(os.environ, PYTHONUNBUFFERED="1")
print("結果の保存先:", RUN_DIR, flush=True)
print("実行ログをリアルタイムに表示します。", flush=True)
process = None
try:
    with CONSOLE_PATH.open("w", encoding="utf-8") as console:
        process = subprocess.Popen(
            [sys.executable, "scripts/run_day2_training.py", "--run-dir", str(RUN_DIR)],
            cwd=REPO, env=training_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                console.write(line)
                console.flush()
            process.wait()
        except KeyboardInterrupt:
            # セルを止めた後も子プロセスだけが学習を続けないよう停止を伝える。
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            try:
                remaining, _ = process.communicate(timeout=15)
            except subprocess.TimeoutExpired:
                process.terminate()
                try:
                    remaining, _ = process.communicate(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    remaining, _ = process.communicate(timeout=5)
            if remaining:
                print(remaining, end="", flush=True)
                console.write(remaining)
                console.flush()
            raise
        finally:
            if process.stdout is not None:
                process.stdout.close()
finally:
    if RUN_DIR.is_dir() and CONSOLE_PATH.is_file():
        destination = RUN_DIR / "console.log"
        CONSOLE_PATH.replace(destination)
        CONSOLE_PATH = destination
    print("コンソールログ:", CONSOLE_PATH, flush=True)

result_path = RUN_DIR / "result.json"
recorded = {}
if result_path.is_file():
    try:
        recorded = json.loads(result_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"result.jsonを読み取れません（{type(exc).__name__}）。コンソールログ: {CONSOLE_PATH}") from exc
    if not isinstance(recorded, dict):
        raise RuntimeError(f"result.jsonが想定した形式ではありません。コンソールログ: {CONSOLE_PATH}")
if process.returncode != 0:
    detail = (f"stage={recorded.get('stage') or '未記録'}, "
              f"error_type={recorded.get('error_type') or '未記録'}, "
              f"error_detail={recorded.get('error_detail') or '詳細はconsole.log/traceback.txtを参照'}")
    raise RuntimeError(f"実行は未完了です（終了コード {process.returncode}）。{detail}。部分結果: {RUN_DIR}。コンソールログ: {CONSOLE_PATH}")
result = recorded
if not (result.get("completed") and result.get("executed_training_myself") and result.get("status") == "completed"):
    raise RuntimeError(f"完了条件を満たしていません。stage={result.get('stage')}、error_type={result.get('error_type')}、error_detail={result.get('error_detail')}。コンソールログ: {CONSOLE_PATH}")
TRAINING_COMPLETED = True
print("学習と前後評価が完了しました。")
print("train:", json.dumps(result["train"], ensure_ascii=False, indent=2))


## 6. 比較と提出用下書きの作成

形式の改善だけで結論を出さず、内容正解率、カテゴリごとの成績、確認要求、打切り、退行した設問の原文も確認します。`final_logged_loss` は最後のログ区間の loss、`train_loss` は学習全体の平均です。別の指標として読みます。

生成された考察に、実際の回答を読んで気づいた点を追記してください。少数の用語・設問で得た結果を一般的な性能保証として扱わないでください。


In [ ]:
from IPython.display import display, Markdown

if not TRAINING_COMPLETED:
    raise RuntimeError("学習・前後評価が完了していないため、完了レポートは生成しません。")
subprocess.run(
    [sys.executable, "scripts/prepare_day2_answer.py", "--run-dir", str(RUN_DIR)],
    cwd=REPO, check=True,
)
ANSWER = REPO / "lectures/02/answer.md"
SUBMISSION_Q3 = REPO / "lectures/02/submission_q3.txt"
if not ANSWER.is_file() or not SUBMISSION_Q3.is_file():
    raise RuntimeError("提出用answer.mdまたはsubmission_q3.txtが見つかりません。レポート生成ログを確認してください。")
REPORT_GENERATED = True
display(Markdown(ANSWER.read_text(encoding="utf-8")))


## 7. 結果 ZIP をダウンロードする

ZIPには、今回の `results/<実行ID>/` 直下のJSONファイル、`console.log`・`traceback.txt`（存在する場合）と、今回生成した `answer.md`・`submission_q3.txt` を含めます。**基盤モデル、adapter、チェックポイントの重みは含めません。** adapter を再利用したい場合は、`RUN_DIR / "adapter"` を別途保存してください。

ダウンロードしたZIPを展開し、`lectures/02/results/<実行ID>/` と、考察を追記した `lectures/02/answer.md`・`lectures/02/submission_q3.txt` を公開 fork の `lec02` に反映します。部分結果の場合は未完了のままであり、学習完了を示す提出物にはなりません。


In [ ]:
import zipfile
from google.colab import files

if "RUN_DIR" not in globals() or not RUN_DIR.is_dir():
    raise RuntimeError("保存された実行フォルダがありません。学習セルのエラーを確認してください。")
json_files = sorted(RUN_DIR.glob("*.json"))
if not json_files:
    raise RuntimeError("ダウンロードできる結果JSONがありません。学習が開始されたか確認してください。")
archive = Path("/content") / f"day2-results-{RUN_DIR.name}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in json_files:
        bundle.write(path, arcname=str(path.relative_to(REPO)))
    for name in ("console.log", "traceback.txt"):
        path = RUN_DIR / name
        if path.is_file():
            bundle.write(path, arcname=str(path.relative_to(REPO)))
    if globals().get("REPORT_GENERATED", False):
        bundle.write(ANSWER, arcname=str(ANSWER.relative_to(REPO)))
        bundle.write(SUBMISSION_Q3, arcname=str(SUBMISSION_Q3.relative_to(REPO)))
print("保存ファイル:", archive)
print("実行状態:", json.loads((RUN_DIR / "result.json").read_text(encoding="utf-8")).get("status"))
files.download(str(archive))


## 参考資料

- [配布ノートブック](lora_format_tuning.ipynb)
- [固定したモデル revision](https://huggingface.co/Qwen/Qwen3.5-0.8B/tree/2fc06364715b967f1860aea9cf38778875588b17)
- [TRL 1.14.2 SFTTrainer](https://huggingface.co/docs/trl/v1.14.2/sft_trainer)
- [PEFT の LoRA](https://huggingface.co/docs/peft/en/conceptual_guides/lora)

`result.json` の `git_commit`、モデル revision、データハッシュ、ライブラリ版を使って、実行した条件を後から確認できます。seedを固定してもGPU学習の完全な再現性を保証するものではありません。
